# Biến đổi hình học trên ảnh — notebook trình bày

Mỗi ô ứng với một mục báo cáo, chạy lần lượt từ trên xuống dưới. Các hàm được gọi ở đây chính là các hàm mà `run_all.py` dùng để sinh `results/`, nên hình và số liệu trùng với báo cáo (riêng thời gian đo có thể dao động nhẹ giữa các lần chạy).

In [ ]:
# Mục 0. Chuẩn bị: thêm thư mục gốc dự án vào sys.path, nạp thư viện
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import cv2, numpy as np
from IPython.display import Image, Markdown, display
from src import experiments as E, transforms as T, images
from src.images import write_data_files
from src.plotting import setup_style
from src.report import system_info

setup_style()
write_data_files()
for k, v in system_info().items():
    print(f'{k}: {v}')

## Mục 1. Quy trình biến đổi hình học với OpenCV — Hình 1, Bảng 1

In [ ]:
r1 = E.make_hinh1()
display(Image(filename=str(r1['path']), width=900))
E.make_bang1()
display(Markdown((ROOT / 'results/tables/Bang1.md').read_text(encoding='utf-8')))

## Mục 2. Co giãn ảnh (cv2.resize) — Hình 2, Bảng 2

In [ ]:
r2 = E.make_hinh2()
display(Image(filename=str(r2['path']), width=1000))
print('PSNR phóng to ×4:', {k: round(v, 2) for k, v in r2['up_psnr'].items()})
print('Độ lệch chuẩn vùng > Nyquist khi thu nhỏ:', {k: round(v, 1) for k, v in r2['alias_std'].items()})

In [ ]:
E.make_bang2()
display(Markdown((ROOT / 'results/tables/Bang2.md').read_text(encoding='utf-8')))

## Mục 3. Tịnh tiến và xoay, chế độ biên — Hình 3

In [ ]:
r3 = E.make_hinh3()
display(Image(filename=str(r3['path']), width=1000))
print('Tỉ lệ ảnh gốc còn lại khi xoay giữ khung (%):', {a: round(v, 1) for a, v in r3['kept_keep'].items()})
print('Kích thước khung mở rộng:', r3['sizes_expand'])

## Mục 4. Biến đổi affine 3 điểm — Hình 4

In [ ]:
r4 = E.make_hinh4()
display(Image(filename=str(r4['path']), width=1000))
print('M =\n', np.round(r4['matrix'], 4))
print('Sai số chiếu điểm:', r4['point_err'])

## Mục 5. Biến đổi phối cảnh 4 điểm — Hình 5

In [ ]:
r5 = E.make_hinh5()
display(Image(filename=str(r5['path']), width=1000))
print('H =\n', np.round(r5['matrix'], 6))
print('Sai số chiếu điểm:', r5['point_err'])

## Mục 6. Ứng dụng: nắn chỉnh ảnh tài liệu chụp nghiêng — Hình 6

In [ ]:
r6 = E.make_hinh6()
display(Image(filename=str(r6['path']), width=1000))
print(f"Sai số góc {r6['corner_err']:.2f} px, PSNR {r6['psnr_auto']:.2f} dB, "
      f"tìm góc {r6['t_detect_ms']:.2f} ms, nắn chỉnh {r6['t_rectify_ms']:.2f} ms")

## Mục 7. Đối chứng: cài đặt tham chiếu (NumPy) so với OpenCV — Hình 7, Bảng 3

Ô này chạy khoảng 20–30 s vì bản NumPy chậm hơn OpenCV hàng trăm lần.

In [ ]:
from compare.compare_ref_vs_cv import run_comparison
cmp = run_comparison()
display(Image(filename=str(cmp['Hinh7']['path']), width=800))
display(Markdown((ROOT / 'results/tables/Bang3.md').read_text(encoding='utf-8')))
for k, v in cmp['quantization'].items():
    print(f"Lượng tử 1/32 px — {k}: {v['float']:.2f} dB → {v['q32']:.2f} dB")

## Mục 8. Số liệu tổng hợp cho báo cáo (report_numbers.md)

In [ ]:
from src.report import write_report
res = {'Hinh2': r2, 'Hinh3': r3, 'Hinh4': r4, 'Hinh5': r5, 'Hinh6': r6, 'Bang2': E.make_bang2()}
display(Markdown(write_report(res, cmp).read_text(encoding='utf-8')))